# 3D-SynTree: Download Mode (Google Colab)

**Role: Raw Data Acquisition & Preservation ONLY.**

This notebook runs in **Google Colab (Internet: ON)**:
1. Streams raw, unprocessed SBDD data (`crossdocked_pocket10.tar.gz`) via accelerated multi-connection aria2 with HTTP range resume.
2. Acquires the canonical Enamine 3D building-block catalog.
3. Cryptographically audits all downloaded raw files using SHA-256.
4. Generates an immutable provenance `raw_manifest.json`.
5. Pushes the **RAW, UNPROCESSED** dataset to Hugging Face.

*Zero RDKit decomposition, zero chemical filtering, and zero tensor construction are performed in Colab.*

In [ ]:
# CELL 1: Hugging Face Authentication
import os
from getpass import getpass

HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    HF_TOKEN = getpass("Enter your Hugging Face WRITE Token: ").strip()

if not HF_TOKEN:
    raise ValueError("A valid Hugging Face WRITE token is required to publish raw sources.")

os.environ["HF_TOKEN"] = HF_TOKEN

from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
username = api.whoami().get("name")
print(f"[Auth] Successfully authenticated as: '{username}'")

# Target repository for RAW unprocessed datasets
TARGET_RAW_REPO_ID = f"{username}/3d-syntree-raw-sources"
print(f"[Target] Raw dataset will be published to: https://huggingface.co/datasets/{TARGET_RAW_REPO_ID}")

In [ ]:
# CELL 2: Environment Setup & Tooling
import os
from pathlib import Path

# 1. Install aria2 for multi-connection resumable downloads
!apt-get update -qq && apt-get install -y -qq aria2
!pip install --quiet huggingface-hub pyarrow pandas

WORKSPACE_DIR = Path("/content/raw_workspace")
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)
print(f"[Setup] Local raw workspace initialized at: {WORKSPACE_DIR}")

In [ ]:
# CELL 3: Resumable Acquisition of Raw Datasets
import os, sys, subprocess, hashlib, time, json
from pathlib import Path

RAW_SOURCES = {
    "crossdocked": {
        "url": "https://huggingface.co/datasets/Yukk1Zz/if3-crossdocked2020/resolve/main/crossdocked_pocket10.tar.gz",
        "filename": "crossdocked_pocket10.tar.gz",
        "expected_sha256": "59416d06c2c366f6e05b91fdff8584f7",
    },
    "enamine_catalog": {
        "url": "https://huggingface.co/datasets/JJKK1212/3d-syntree-multidataset/resolve/main/enamine_3d_subset.parquet",
        "filename": "enamine_3d_subset.parquet",
        "expected_sha256": None,
    },
}

def compute_sha256(path: Path) -> str:
    hasher = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(4 * 1024 * 1024):
            hasher.update(chunk)
    return hasher.hexdigest()

print("=" * 70)
print("STARTING RESUMABLE RAW DATASET ACQUISITION")
print("=" * 70)

for key, cfg in RAW_SOURCES.items():
    dest_path = WORKSPACE_DIR / cfg["filename"]
    print(f"\n[Check] {cfg['filename']}...")
    
    already_valid = False
    if dest_path.exists():
        current_sha = compute_sha256(dest_path)
        if cfg["expected_sha256"] and current_sha.lower() == cfg["expected_sha256"].lower():
            print(f"  -> Verified existing file (SHA: {current_sha[:12]}...). Skipping download.")
            already_valid = True
        elif not cfg["expected_sha256"] and dest_path.stat().st_size > 1024:
            print(f"  -> Verified existing file ({dest_path.stat().st_size} bytes). Skipping download.")
            already_valid = True

    if not already_valid:
        print(f"  -> Downloading from: {cfg['url']}")
        # aria2c with -c allows instant byte-level resume across timeouts
        cmd = [
            "aria2c",
            "-x", "16",
            "-s", "16",
            "-k", "1M",
            "-c",
            "-d", str(WORKSPACE_DIR),
            "-o", cfg["filename"],
            cfg["url"],
        ]
        subprocess.run(cmd, check=True)
        
        new_sha = compute_sha256(dest_path)
        if cfg["expected_sha256"] and new_sha.lower() != cfg["expected_sha256"].lower():
            raise IOError(f"SHA-256 mismatch for {dest_path.name}!")
        print(f"  -> Download complete and verified (SHA: {new_sha[:12]}...).")

# ------------------------------------------------------------------------------
# Write Immutable Raw Manifest & Provenance
# ------------------------------------------------------------------------------
print("\n[Manifest] Generating immutable raw provenance manifest...")
manifest_records = {}
for key, cfg in RAW_SOURCES.items():
    p = WORKSPACE_DIR / cfg["filename"]
    manifest_records[key] = {
        "filename": cfg["filename"],
        "size_bytes": p.stat().st_size,
        "sha256": compute_sha256(p),
        "source_url": cfg["url"],
    }

raw_manifest = {
    "dataset_type": "3d_syntree_raw_unprocessed",
    "version": 1,
    "created_at": time.time(),
    "files": manifest_records,
}

manifest_path = WORKSPACE_DIR / "raw_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(raw_manifest, f, indent=2, sort_keys=True)

print(f"[Manifest] Written to {manifest_path}:")
print(json.dumps(raw_manifest, indent=2))

In [ ]:
# CELL 4: Publish Raw Unprocessed Repository to Hugging Face
from pathlib import Path
from huggingface_hub import HfApi

print("=" * 70)
print(f"PUBLISHING RAW DATASET TO HUGGING FACE: {TARGET_RAW_REPO_ID}")
print("=" * 70)

api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=TARGET_RAW_REPO_ID, repo_type="dataset", exist_ok=True)

readme_text = f"""---
license: mit
task_categories:
- graph-ml
tags:
- biology
- chemistry
- raw-sbdd-sources
pretty_name: 3D-SynTree Raw Unprocessed Sources
size_categories:
- 10K<n<100K
---

# 3D-SynTree: Raw SBDD Source Archives

This repository contains **unprocessed, raw structural biology archives**.
No chemical filtering, pocket trimming, retrosynthetic decomposition, or tensor conversion
has been performed on these files.

### Raw Contents:
- `crossdocked_pocket10.tar.gz`: Complete, unmodified CrossDocked2020 10Å pocket complexes.
- `enamine_3d_subset.parquet`: Certified canonical Enamine building blocks catalog.
- `raw_manifest.json`: Cryptographically signed provenance and integrity registry.

### Consumption:
Mount this raw repository directly into Kaggle to run **Craft Mode** for full-scale trajectory construction.
"""
(WORKSPACE_DIR / "README.md").write_text(readme_text, encoding="utf-8")

print("Uploading raw files to Hugging Face Hub (streaming via Git-LFS)...")
api.upload_folder(
    folder_path=str(WORKSPACE_DIR),
    repo_id=TARGET_RAW_REPO_ID,
    repo_type="dataset",
    commit_message="Add verified raw SBDD source archives and provenance manifest",
)

print("\n" + "=" * 70)
print(f"SUCCESS: RAW DATASET PUBLISHED:")
print(f"https://huggingface.co/datasets/{TARGET_RAW_REPO_ID}")
print("=" * 70)
print("\nNext step: Import this repository into Kaggle as a dataset and run Craft Mode!")